In [1]:
import cv2
import os
import torch

import albumentations as A
import numpy as np
import torch.nn as nn
import torch.optim as optim

from PIL import Image
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision.io import decode_image
from torchvision.transforms import ToPILImage, v2

# Ensure we are using the GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Import the dataset
from dataset import AI4MarsDataset

img_transform = A.Compose([
    A.PadIfNeeded(min_height=512, min_width=512, fill_mask=255, border_mode=0),
    A.RandomCrop(512, 512),
    A.RandomRotate90
])

# Change the images and labels paths here to match what's on your computer
ai4mars = AI4MarsDataset(
    "/home/kozakw/cs400/cs400_image_segmentation/data/train/images",
    "/home/kozakw/cs400/cs400_image_segmentation/data/train/labels",
    transform=img_transform
)

train, test = random_split(ai4mars, [0.8, 0.2])
train_dataloader = DataLoader(train, batch_size=8, shuffle=True)
test_dataloader = DataLoader(test, batch_size=1, shuffle=True)


In [ ]:
# Import the model
from model import UNet

model = UNet().to(device)
loss_fn = nn.CrossEntropyLoss(ignore_index=255)
opt = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10
for epoch in range(num_epochs):

    print(f"EPOCH {epoch+1}:")
    model.train()
    
    running_loss = 0.0
    
    for i, data in enumerate(train_dataloader):
        input, label = data
        input = input.to(device)
        label = label.to(device)

        if (label != 255).sum() == 0:
            continue

        opt.zero_grad()
        output = model(input)
        loss = loss_fn(output, label)
        loss.backward()
        opt.step()

        running_loss += loss.item()
        if (i + 1) % 10 == 0:
            print(f"  Batch {i + 1} loss: {running_loss / 10:.4f}")
            running_loss = 0.0

torch.save(model.state_dict(), 'output/unet_weights.pth')

EPOCH 1:
  Batch 10 loss: 1.4724
  Batch 20 loss: 1.2488
  Batch 30 loss: 1.2901
  Batch 40 loss: 1.1889
  Batch 50 loss: 1.0972
  Batch 60 loss: 1.2636
  Batch 70 loss: 1.1432
  Batch 80 loss: 1.0729


KeyboardInterrupt: 